<a href="https://colab.research.google.com/github/RexTabachnick/Machine-Unlearning-In-Applied-Contexts/blob/main/VAE_On_Individual_Celebrities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, torch, pandas as pd
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
from diffusers import AutoencoderKL

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load celebrity dataset
import kagglehub
path = kagglehub.dataset_download("vishesh1412/celebrity-face-image-dataset")
dataset_path = os.path.join(path, "Celebrity Faces Dataset")

records = []
for celebrity in sorted(os.listdir(dataset_path)):
    celeb_dir = os.path.join(dataset_path, celebrity)
    if not os.path.isdir(celeb_dir):
        continue
    for img_file in os.listdir(celeb_dir):
        if img_file.lower().endswith((".jpg", ".jpeg", ".png")):
            records.append({"identity": celebrity, "filepath": os.path.join(celeb_dir, img_file)})

import pandas as pd
df = pd.DataFrame(records)
identity_names = sorted(df["identity"].unique())
print(f"Total images: {len(df)}")
print(f"Celebrities: {identity_names}")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Using Colab cache for faster access to the 'celebrity-face-image-dataset' dataset.
Total images: 1800
Celebrities: ['Angelina Jolie', 'Brad Pitt', 'Denzel Washington', 'Hugh Jackman', 'Jennifer Lawrence', 'Johnny Depp', 'Kate Winslet', 'Leonardo DiCaprio', 'Megan Fox', 'Natalie Portman', 'Nicole Kidman', 'Robert Downey Jr', 'Sandra Bullock', 'Scarlett Johansson', 'Tom Cruise', 'Tom Hanks', 'Will Smith']


In [2]:
# INSTALLING A PRETRAINED VAE

sd_vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
sd_vae.eval()

transform = transforms.Compose([
    transforms.Resize((256, 256)),   # SD VAE works better at 256
    transforms.ToTensor(),
])

def encode(imgs):
    imgs_scaled = imgs * 2 - 1
    with torch.no_grad():
        latent = sd_vae.encode(imgs_scaled).latent_dist.mean
    return latent

def decode(latent):
    with torch.no_grad():
        recon = sd_vae.decode(latent).sample
    return ((recon + 1) / 2).clamp(0, 1)

print("SD VAE ready")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

SD VAE ready


In [3]:
from torch.utils.data import Dataset

class CelebDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.df.loc[idx, "filepath"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.df.loc[idx, "identity"]

# Pick forget target
FORGET_CELEB = identity_names[0]   # change to any name in the list
print(f"Forget target: {FORGET_CELEB}")

forget_df = df[df["identity"] == FORGET_CELEB].reset_index(drop=True)
retain_df  = df[df["identity"] != FORGET_CELEB].reset_index(drop=True)

forget_loader = DataLoader(CelebDataset(forget_df, transform), batch_size=8, shuffle=False)
retain_loader  = DataLoader(CelebDataset(retain_df,  transform), batch_size=8, shuffle=False)

# Encode all images and compute direction
print("Encoding forget set...")
forget_latents = torch.cat([encode(imgs.to(device)) for imgs, _ in forget_loader], dim=0)

print("Encoding retain set...")
retain_latents = torch.cat([encode(imgs.to(device)) for imgs, _ in retain_loader], dim=0)

mean_forget = forget_latents.mean(dim=0, keepdim=True)
mean_retain  = retain_latents.mean(dim=0, keepdim=True)
celeb_direction = mean_forget - mean_retain

print(f"Forget images encoded: {forget_latents.shape[0]}")
print(f"Retain images encoded: {retain_latents.shape[0]}")
print(f"Celebrity direction norm: {celeb_direction.norm():.4f}")

Forget target: Angelina Jolie
Encoding forget set...
Encoding retain set...
Forget images encoded: 100
Retain images encoded: 1700
Celebrity direction norm: 54.3056


In [ ]:
import copy

# Save original on CPU — only pull to GPU when needed
sd_vae_original = copy.deepcopy(sd_vae).cpu()
sd_vae_original.eval()
for p in sd_vae_original.parameters():
    p.requires_grad = False

# Also reduce batch size — 256x256 is huge
# Change your DataLoader to use smaller batches
forget_loader_small = DataLoader(CelebDataset(forget_df, transform), batch_size=2, shuffle=True)
retain_loader_small = DataLoader(CelebDataset(retain_df, transform), batch_size=2, shuffle=True)

# Clear GPU memory from previous failed run
torch.cuda.empty_cache()

unlearn_optimizer = torch.optim.Adam(sd_vae.decoder.parameters(), lr=5e-6)

STEPS = 150
print("=== UNLEARNING ANGELINA FROM SD VAE ===")

for step in range(STEPS):
    sd_vae.train()

    # Forget step
    imgs, _ = next(iter(forget_loader_small))
    imgs = imgs.to(device)
    with torch.no_grad():
        latents = encode(imgs)
        shuffled = latents[torch.randperm(latents.size(0))]
        # Pull original to GPU briefly just for target generation
        sd_vae_original.to(device)
        target = sd_vae_original.decode(shuffled).sample
        sd_vae_original.cpu()  # immediately back to CPU
        torch.cuda.empty_cache()

    recon = sd_vae.decode(latents).sample
    loss_forget = torch.nn.functional.mse_loss(recon, target.detach())

    # Retain step
    r_imgs, _ = next(iter(retain_loader_small))
    r_imgs = r_imgs.to(device)
    with torch.no_grad():
        r_latents = encode(r_imgs)
        sd_vae_original.to(device)
        good_target = sd_vae_original.decode(r_latents).sample
        sd_vae_original.cpu()
        torch.cuda.empty_cache()

    r_recon = sd_vae.decode(r_latents).sample
    loss_retain = torch.nn.functional.mse_loss(r_recon, good_target.detach())

    loss = loss_forget + 5.0 * loss_retain
    unlearn_optimizer.zero_grad()
    loss.backward()
    unlearn_optimizer.step()

    if step % 30 == 0 or step == STEPS - 1:
        print(f"Step {step:3d} | Forget: {loss_forget.item():.5f} | Retain: {loss_retain.item():.5f}")



sd_vae_unlearned_state = copy.deepcopy(sd_vae.state_dict())
sd_vae_original_state  = copy.deepcopy(sd_vae_original.state_dict())
print("States saved to memory.")

print("Done.")

=== UNLEARNING ANGELINA FROM SD VAE ===
Step   0 | Forget: 0.00000 | Retain: 0.00000
Step  30 | Forget: 0.37591 | Retain: 0.00030
Step  60 | Forget: 0.00111 | Retain: 0.00065
Step  90 | Forget: 0.00244 | Retain: 0.00191
Step 120 | Forget: 0.00215 | Retain: 0.00098


In [5]:
sd_vae.eval()

class CelebDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.df.loc[idx, "filepath"]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, 0

transform_small = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

forget_loader_vis = DataLoader(CelebDataset(forget_df, transform_small), batch_size=4, shuffle=False)
retain_loader_vis = DataLoader(CelebDataset(retain_df, transform_small), batch_size=4, shuffle=False)

test_forget, _ = next(iter(forget_loader_vis))
test_retain, _  = next(iter(retain_loader_vis))
test_forget = test_forget.to(device)
test_retain  = test_retain.to(device)

# AFTER — current model in memory (just finished unlearning)
forget_after = decode(encode(test_forget))
retain_after  = decode(encode(test_retain))

# BEFORE — reload original using the copy we kept in memory
sd_vae.load_state_dict(sd_vae_original_state)
sd_vae.eval()
forget_before = decode(encode(test_forget))
retain_before  = decode(encode(test_retain))

# Restore unlearned weights
sd_vae.load_state_dict(sd_vae_unlearned_state)
sd_vae.eval()

# Plot
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i in range(4):
    axes[0,i].imshow(test_forget[i].cpu().permute(1,2,0).clamp(0,1))
    axes[0,i].axis('off')
    if i == 0: axes[0,i].set_title('Original input', fontsize=9)
    axes[1,i].imshow(forget_before[i].cpu().permute(1,2,0).clamp(0,1))
    axes[1,i].axis('off')
    if i == 0: axes[1,i].set_title('BEFORE unlearning', fontsize=9)
    axes[2,i].imshow(forget_after[i].cpu().permute(1,2,0).clamp(0,1))
    axes[2,i].axis('off')
    if i == 0: axes[2,i].set_title('AFTER unlearning', fontsize=9)
plt.suptitle("Angelina Jolie — Before vs After Unlearning", fontsize=12)
plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    axes2[0,i].imshow(retain_before[i].cpu().permute(1,2,0).clamp(0,1))
    axes2[0,i].axis('off')
    if i == 0: axes2[0,i].set_title('BEFORE', fontsize=9)
    axes2[1,i].imshow(retain_after[i].cpu().permute(1,2,0).clamp(0,1))
    axes2[1,i].axis('off')
    if i == 0: axes2[1,i].set_title('AFTER', fontsize=9)
plt.suptitle("Other celebrities — should be unaffected", fontsize=12)
plt.tight_layout()
plt.show()

fm_b = F.mse_loss(forget_before, test_forget).item()
fm_a = F.mse_loss(forget_after,  test_forget).item()
rm_b = F.mse_loss(retain_before, test_retain).item()
rm_a = F.mse_loss(retain_after,  test_retain).item()

print(f"\n{'':25} {'Before':>10} {'After':>10}")
print(f"{'Angelina MSE':25} {fm_b:>10.5f} {fm_a:>10.5f}")
print(f"{'Others MSE':25} {rm_b:>10.5f} {rm_a:>10.5f}")
print(f"\nForget degraded:  {(fm_a/fm_b-1)*100:.1f}%  ← want BIG")
print(f"Retain changed:   {(rm_a/rm_b-1)*100:+.1f}%  ← want SMALL")

FileNotFoundError: [Errno 2] No such file or directory: '/content/sd_vae_orig.pt'